In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = np.load("spectrum_sensing_features.npz")

X = data["X"]
y = data["y"]
snr = data["snr"]

stratify_groups = np.array([
    f"{label}_{s}"
    for label, s in zip(y, snr)
])

indices = np.arange(len(y))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=stratify_groups
)

temp_groups = np.array([
    f"{y[i]}_{snr[i]}"
    for i in temp_idx
])

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_groups
)

X_train, X_temp = X[train_idx], X[temp_idx]
y_train, y_temp = y[train_idx], y[temp_idx]

X_val, X_test = X[temp_idx], X[test_idx]

In [3]:
X_train, y_train = X[train_idx], y[train_idx]
X_temp, y_temp = X[temp_idx], y[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_groups
)

X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

snr_train = snr[train_idx]
snr_val = snr[val_idx]
snr_test = snr[test_idx]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (9800, 13)
Validation: (2100, 13)
Testing: (2100, 13)


In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")

Feature scaling completed!


In [16]:
import joblib

joblib.dump(scaler, "logistic_regression_scaler.pkl")

print("Original fitted scaler saved successfully.")

Original fitted scaler saved successfully.


In [5]:
# Initialise Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

# Train the model
lr_model.fit(X_train_scaled, y_train)

print("Logistic Regression training completed!")

Logistic Regression training completed!


In [17]:
import joblib

# Save trained model
joblib.dump(lr_model, "logistic_regression_model.pkl")

# Save calibrated threshold
np.save("logistic_regression_threshold.npy", lr_threshold)

print("Logistic Regression model and threshold saved.")
print("Threshold:", lr_threshold)

Logistic Regression model and threshold saved.
Threshold: 0.7525429211626531


In [6]:
# Predict class labels
y_val_pred = lr_model.predict(X_val_scaled)

# Predict probabilities of occupied channel (H1)
y_val_prob = lr_model.predict_proba(X_val_scaled)[:, 1]

print("Validation predictions generated!")
print("Predictions shape:", y_val_pred.shape)
print("Probability shape:", y_val_prob.shape)

Validation predictions generated!
Predictions shape: (2100,)
Probability shape: (2100,)


In [7]:
from sklearn.metrics import accuracy_score

val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Preliminary Validation Accuracy: {val_accuracy:.4f}")

Preliminary Validation Accuracy: 0.7214


In [8]:
target_pfa = 0.05

# Select H0 validation probabilities
h0_val_prob = y_val_prob[y_val == 0]

# Calculate threshold using the 95th percentile
lr_threshold = np.quantile(
    h0_val_prob,
    1 - target_pfa,
    method="higher"
)

print("Logistic Regression threshold:", lr_threshold)

Logistic Regression threshold: 0.7525429211626531


In [9]:
# Predict using calibrated threshold
y_val_calibrated = (
    y_val_prob > lr_threshold
).astype(int)

# Calculate validation false alarm probability
val_h0_mask = y_val == 0

val_pfa = np.mean(
    y_val_calibrated[val_h0_mask] == 1
)

print(f"Target Pfa: {target_pfa:.4f}")
print(f"Actual validation Pfa: {val_pfa:.4f}")

Target Pfa: 0.0500
Actual validation Pfa: 0.0495


In [10]:
val_h1_mask = y_val == 1

val_pd = np.mean(
    y_val_calibrated[val_h1_mask] == 1
)

print(f"Validation Pd: {val_pd:.4f}")
print(f"Validation Pm: {1 - val_pd:.4f}")

Validation Pd: 0.4048
Validation Pm: 0.5952


In [11]:
# Predict probabilities on test dataset
y_test_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Apply calibrated threshold
y_test_pred = (
    y_test_prob > lr_threshold
).astype(int)

print("Test predictions generated!")

Test predictions generated!


In [12]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1]
).ravel()

test_pd = tp / (tp + fn)
test_pfa = fp / (fp + tn)
test_pm = fn / (fn + tp)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("LOGISTIC REGRESSION - TEST RESULTS")
print("-----------------------------------")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Probability of Detection (Pd): {test_pd:.4f}")
print(f"Probability of False Alarm (Pfa): {test_pfa:.4f}")
print(f"Probability of Missed Detection (Pm): {test_pm:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

LOGISTIC REGRESSION - TEST RESULTS
-----------------------------------
Accuracy: 0.6719
Probability of Detection (Pd): 0.3838
Probability of False Alarm (Pfa): 0.0400
Probability of Missed Detection (Pm): 0.6162

Confusion Matrix:
[[1008   42]
 [ 647  403]]


In [13]:
from sklearn.metrics import roc_auc_score

lr_auc = roc_auc_score(y_test, y_test_prob)

print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")

Logistic Regression ROC-AUC: 0.8314


In [14]:
lr_results = []

for s in np.unique(snr_test):

    mask = snr_test == s

    true_labels = y_test[mask]
    predictions = y_test_pred[mask]

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    lr_results.append({
        "SNR": s,
        "Pd": tp / (tp + fn),
        "Pfa": fp / (fp + tn),
        "Pm": fn / (fn + tp)
    })

lr_df = pd.DataFrame(lr_results)

print(lr_df.round(4))

   SNR      Pd     Pfa      Pm
0  -20  0.0733  0.0667  0.9267
1  -15  0.0400  0.0667  0.9600
2  -10  0.1333  0.1067  0.8667
3   -5  0.1400  0.0400  0.8600
4    0  0.3667  0.0000  0.6333
5    5  0.9467  0.0000  0.0533
6   10  0.9867  0.0000  0.0133


In [15]:
lr_df.to_csv(
    "logistic_regression_results.csv",
    index=False
)

lr_summary = pd.DataFrame([{
    "Accuracy": test_accuracy,
    "Pd": test_pd,
    "Pfa": test_pfa,
    "Pm": test_pm,
    "ROC_AUC": lr_auc
}])

lr_summary.to_csv(
    "logistic_regression_summary.csv",
    index=False
)

print("Logistic Regression results saved!")

Logistic Regression results saved!
